# Train LeRobot ACT Policies on Google Colab (GPU)

Trains one **ACT (Action Chunking with Transformers)** policy per manipulation skill, using **LeRobot 0.6.1**.

This version takes **one combined zip** containing every skill's dataset as a top-level folder
(`open_drawer/`, `pick_plate/`, `pick_mug/`, `pick_bottle/`, `place_plate/`, `pour_water/`, each a real
LeRobot v3.0 `LeRobotDataset` produced by `scripts/build_clean_dataset.py`). It auto-discovers whichever
skill folders are actually inside the zip and trains a separate policy for each - no per-skill upload,
no manual list of filenames to edit.

### Step 1: Set Runtime to GPU
Go to **Runtime -> Change runtime type -> T4 GPU (or any GPU) -> Save**.

### Step 2: Upload the combined dataset zip
Drag `data/butler_demos_all.zip` from your local repo into the Colab file explorer on the left
(or upload to Drive and mount it, then point `ZIP_PATH` below at that location).

In [ ]:
# 0. Configuration

ZIP_PATH = "/content/butler_demos_all.zip"  # where you uploaded the combined dataset zip
EXTRACT_ROOT = "/content/dataset"

DEFAULT_STEPS = 10000
DEFAULT_BATCH_SIZE = 8

# Optional: exclude specific skills even if their folder is present in the zip
# (e.g. to re-run just one skill without retraining everything).
SKIP_SKILLS = set()  # e.g. {"open_drawer"} to skip a skill you already trained

# Optional: override steps/batch_size for specific skills instead of the defaults above.
PER_SKILL_OVERRIDES = {
    # "pour_water": {"steps": 15000, "batch_size": 8},
}

In [ ]:
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/lerobot_outputs"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 1. Verify GPU availability
!nvidia-smi

Tue Sep 22 10:36:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# 2. Install LeRobot with training extras
!pip install --upgrade pip
!pip install "lerobot[training]>=0.6.1"

In [ ]:
# 3. Extract the combined zip once, then auto-discover which skill folders it contains.
# A folder counts as a skill dataset only if it has meta/info.json - anything else in the
# zip (stray files, __MACOSX/, etc.) is ignored rather than guessed at.
import os

assert os.path.exists(ZIP_PATH), f"{ZIP_PATH} not found - upload data/butler_demos_all.zip first."
!unzip -o -q "{ZIP_PATH}" -d "{EXTRACT_ROOT}"

dataset_roots = {}
for name in sorted(os.listdir(EXTRACT_ROOT)):
    candidate = os.path.join(EXTRACT_ROOT, name)
    if os.path.isdir(candidate) and os.path.isfile(os.path.join(candidate, "meta", "info.json")):
        dataset_roots[name] = candidate

assert dataset_roots, f"No skill datasets found under {EXTRACT_ROOT} - check the zip contents."
print(f"Discovered {len(dataset_roots)} skill dataset(s): {list(dataset_roots)}")
for skill, root in dataset_roots.items():
    print(f"  [{skill}] -> {root}  (meta files: {os.listdir(os.path.join(root, 'meta'))})")

Discovered 6 skill dataset(s): ['open_drawer', 'pick_bottle', 'pick_mug', 'pick_plate', 'place_plate', 'pour_water']
  [open_drawer] -> /content/dataset/open_drawer  (meta files: ['butler_episodes.json', 'info.json', 'stats.json', 'episodes', 'tasks.parquet'])
  [pick_bottle] -> /content/dataset/pick_bottle  (meta files: ['butler_episodes.json', 'info.json', 'stats.json', 'episodes', 'tasks.parquet'])
  [pick_mug] -> /content/dataset/pick_mug  (meta files: ['butler_episodes.json', 'info.json', 'stats.json', 'episodes', 'tasks.parquet'])
  [pick_plate] -> /content/dataset/pick_plate  (meta files: ['butler_episodes.json', 'info.json', 'stats.json', 'episodes', 'tasks.parquet'])
  [place_plate] -> /content/dataset/place_plate  (meta files: ['butler_episodes.json', 'info.json', 'stats.json', 'episodes', 'tasks.parquet'])
  [pour_water] -> /content/dataset/pour_water  (meta files: ['butler_episodes.json', 'info.json', 'stats.json', 'episodes', 'tasks.parquet'])


In [ ]:
# 4. Apply SKIP_SKILLS and build the final per-skill training plan.

skills_to_train = {
    skill: root for skill, root in dataset_roots.items() if skill not in SKIP_SKILLS
}
assert skills_to_train, "Every discovered skill is in SKIP_SKILLS - nothing left to train."

plan = {}
for skill in skills_to_train:
    override = PER_SKILL_OVERRIDES.get(skill, {})
    plan[skill] = {
        "root": skills_to_train[skill],
        "steps": override.get("steps", DEFAULT_STEPS),
        "batch_size": override.get("batch_size", DEFAULT_BATCH_SIZE),
    }

print("Training plan:")
for skill, cfg in plan.items():
    print(f"  [{skill}] steps={cfg['steps']} batch_size={cfg['batch_size']} root={cfg['root']}")

Training plan:
  [open_drawer] steps=10000 batch_size=8 root=/content/dataset/open_drawer
  [pick_bottle] steps=10000 batch_size=8 root=/content/dataset/pick_bottle
  [pick_mug] steps=10000 batch_size=8 root=/content/dataset/pick_mug
  [pick_plate] steps=10000 batch_size=8 root=/content/dataset/pick_plate
  [place_plate] steps=10000 batch_size=8 root=/content/dataset/place_plate
  [pour_water] steps=10000 batch_size=8 root=/content/dataset/pour_water


In [ ]:
# 5. (Recommended) Validate each dataset against the Butler schema before spending
# GPU time on it - catches a malformed export early rather than mid-training.
import subprocess

!git clone --depth 1 https://github.com/MUHAMMAD-AZEEM-AZAM/Butler-AI.git /content/repo
%cd /content/repo

for skill, cfg in plan.items():
    print(f"\n=== validating {skill} ===")
    result = subprocess.run(
        ["python", "-m", "stage3_policy.learned.validate_dataset", "--root", cfg["root"], "--min-successful-episodes", "1"],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(f"WARNING: {skill} dataset did not fully validate - inspect the report above before trusting the checkpoint.")

fatal: destination path '/content/repo' already exists and is not an empty directory.
/content/repo

=== validating open_drawer ===
dataset: /content/dataset/open_drawer
status: VALID
  total_episodes: 8
  total_frames: 1395
  episodes_per_skill: {'open_drawer': 8}
  successful_per_skill: {'open_drawer': 8}
  episodes_per_split: {'train': 6, 'val': 2}


=== validating pick_bottle ===
dataset: /content/dataset/pick_bottle
status: VALID
  total_episodes: 6
  total_frames: 728
  episodes_per_skill: {'pick': 6}
  successful_per_skill: {'pick': 6}
  episodes_per_split: {'train': 4, 'val': 2}


=== validating pick_mug ===
dataset: /content/dataset/pick_mug
status: VALID
  total_episodes: 6
  total_frames: 1062
  episodes_per_skill: {'pick': 6}
  successful_per_skill: {'pick': 6}
  episodes_per_split: {'train': 4, 'val': 2}


=== validating pick_plate ===
dataset: /content/dataset/pick_plate
status: VALID
  total_episodes: 6
  total_frames: 598
  episodes_per_skill: {'pick': 6}
  successful_p

In [ ]:
# 6. Launch ACT training on GPU, one policy per skill in the plan.
# Sequential, not parallel - each run gets the full GPU, and a failure in one
# skill's training does not stop the others from being attempted.

for skill, cfg in plan.items():
    # Output directory now points to Google Drive
    output_dir = f"{DRIVE_OUTPUT_ROOT}/train/act_{skill}"
    print(f"\n{'='*70}\nTraining '{skill}' -> {output_dir}\n{'='*70}")

    # Check if a checkpoint exists for this skill in the drive and use it to resume
    last_checkpoint_path = f"{output_dir}/checkpoints/last/pretrained_model"
    if os.path.exists(last_checkpoint_path):
        print(f"Resuming training for '{skill}' from checkpoint: {last_checkpoint_path}")
        checkpoint_arg = f"--pretrained_model_path={last_checkpoint_path}"
    else:
        print(f"No checkpoint found for '{skill}' at {last_checkpoint_path}. Starting new training.")
        checkpoint_arg = ""

    # Construct the full shell command as a Python f-string first
    command = f"""lerobot-train \
      --dataset.repo_id=local/butler_demos_{skill} \
      --dataset.root={cfg['root']} \
      --policy.type=act \
      --policy.device=cuda \
      --policy.push_to_hub=false \
      --output_dir={output_dir} \
      {checkpoint_arg} \
      --job_name=act_{skill} \
      --steps={cfg['steps']} \
      --batch_size={cfg['batch_size']} \
      --save_freq=2500 \
      --seed=1000 \
      --wandb.enable=false"""

    # Execute the constructed command
    !{command}


Training 'open_drawer' -> /content/drive/MyDrive/lerobot_outputs/train/act_open_drawer
Resuming training for 'open_drawer' from checkpoint: /content/drive/MyDrive/lerobot_outputs/train/act_open_drawer/checkpoints/last/pretrained_model
usage: lerobot-train [-h] [--config_path str] [--dataset str]
                     [--dataset.repo_id str] [--dataset.repo_type str]
                     [--dataset.root str] [--dataset.episodes str]
                     [--image_transforms str] [--dataset.image_transforms str]
                     [--dataset.image_transforms.enable str]
                     [--dataset.image_transforms.max_num_transforms str]
                     [--dataset.image_transforms.random_order str]
                     [--dataset.image_transforms.tfs str]
                     [--dataset.revision str]
                     [--dataset.use_imagenet_stats str]
                     [--dataset.video_backend str]
                     [--dataset.return_uint8 str]
                     [-

In [ ]:
# 7. Package and download every trained checkpoint that actually completed.
import os
from google.colab import files

for skill in plan:
    ckpt_dir = f"/content/outputs/train/act_{skill}/checkpoints/last/pretrained_model"
    if not os.path.isdir(ckpt_dir):
        print(f"[{skill}] no checkpoint found at {ckpt_dir} - training likely did not complete, skipping download.")
        continue
    zip_name = f"/content/trained_act_{skill}_checkpoint.zip"
    !zip -r {zip_name} {ckpt_dir}
    files.download(zip_name)
    print(f"[{skill}] checkpoint download initiated: {zip_name}")

[open_drawer] no checkpoint found at /content/outputs/train/act_open_drawer/checkpoints/last/pretrained_model - training likely did not complete, skipping download.
[pick_bottle] no checkpoint found at /content/outputs/train/act_pick_bottle/checkpoints/last/pretrained_model - training likely did not complete, skipping download.
[pick_mug] no checkpoint found at /content/outputs/train/act_pick_mug/checkpoints/last/pretrained_model - training likely did not complete, skipping download.
[pick_plate] no checkpoint found at /content/outputs/train/act_pick_plate/checkpoints/last/pretrained_model - training likely did not complete, skipping download.
[place_plate] no checkpoint found at /content/outputs/train/act_place_plate/checkpoints/last/pretrained_model - training likely did not complete, skipping download.
[pour_water] no checkpoint found at /content/outputs/train/act_pour_water/checkpoints/last/pretrained_model - training likely did not complete, skipping download.


In [ ]:
# 7. Checkpoints are now saved directly to Google Drive. No explicit download needed.
# If you wish to zip them within Drive, you can uncomment the zip command below.

import os

for skill in plan:
    # The checkpoint directory is now within Google Drive
    ckpt_dir = f"{DRIVE_OUTPUT_ROOT}/train/act_{skill}/checkpoints/last/pretrained_model"
    if not os.path.isdir(ckpt_dir):
        print(f"[{skill}] no checkpoint found at {ckpt_dir} - training likely did not complete, skipping.")
        continue

    print(f"[{skill}] checkpoint saved to Google Drive at: {ckpt_dir}")
    # If you still want to create a zip of the checkpoint within your Drive,
    # uncomment the following lines:
    # zip_name = f"{DRIVE_OUTPUT_ROOT}/trained_act_{skill}_checkpoint.zip"
    # !zip -r {zip_name} {ckpt_dir}
    # print(f"[{skill}] checkpoint zipped within Google Drive: {zip_name}")

[open_drawer] checkpoint saved to Google Drive at: /content/drive/MyDrive/lerobot_outputs/train/act_open_drawer/checkpoints/last/pretrained_model
[pick_bottle] checkpoint saved to Google Drive at: /content/drive/MyDrive/lerobot_outputs/train/act_pick_bottle/checkpoints/last/pretrained_model
[pick_mug] checkpoint saved to Google Drive at: /content/drive/MyDrive/lerobot_outputs/train/act_pick_mug/checkpoints/last/pretrained_model
[pick_plate] checkpoint saved to Google Drive at: /content/drive/MyDrive/lerobot_outputs/train/act_pick_plate/checkpoints/last/pretrained_model
[place_plate] checkpoint saved to Google Drive at: /content/drive/MyDrive/lerobot_outputs/train/act_place_plate/checkpoints/last/pretrained_model
[pour_water] checkpoint saved to Google Drive at: /content/drive/MyDrive/lerobot_outputs/train/act_pour_water/checkpoints/last/pretrained_model
